<a href="https://colab.research.google.com/github/di-yeferson/analitica-empresarial-integrada/blob/main/LAB_D10_AEI_PSAYAN_2026_02_QUISPE_DIEGO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO DIRIGIDO N.° 10 — Pronóstico de demanda

**Semana 10 · TECSUP 2026-II**
**Docente:** PILAR ROCIO SAYAN MEJIA
**Dataset real:** Online Retail II — UCI Machine Learning Repository (ID 502)
**Objetivo:** construir un pronóstico semanal de ventas, seleccionar modelo y política de cantidad con validación temporal sin información futura, evaluarlos una sola vez y decidir si conviene adoptarlos.

> El notebook termina en una **decisión fundamentada** (adoptar, mantener en evaluación o no adoptar), no solo en el modelo con menor error.

## Trabajo individual y entrega

Esta práctica dirigida es **individual**. Ejecute el notebook en **su propio Google Colab**, verifique los resultados y responda con sus propias interpretaciones. No se requiere informe separado: las respuestas, el cierre ejecutivo y las conclusiones quedan en este notebook.

**Al final de la sesión:**
1. Suba este notebook, ejecutado y con los retos resueltos, a la carpeta `semana10/` de su repositorio personal de GitHub.
2. Conserve la versión en su propio Google Colab y compártala con permiso de lectura.
3. En la **actividad de la semana** presente **ambos URLs**: el enlace directo al archivo en GitHub y el enlace a su Colab.

**Identificación**
- Nombre: Diego Yeferson Quispe Huamani
- Sección: C24 6to A
- URL de GitHub: https://github.com/di-yeferson/analitica-empresarial-integrada/blob/main/semana10/LAB_D10_AEI_PSAYAN_2026_02_ESTUDIANTE.ipynb
- URL de Colab: https://drive.google.com/file/d/1pqRQ3YGqqTKgYXXjk5c1PI26PZ5blGrU/view?usp=sharing

- Uso de IA declarado: Se usó la IA para interpretar algunos resultados y entender los conceptos del laboratorio.

# 1. Pregunta de negocio

**¿Cuántas unidades debería planificar la empresa para las próximas semanas, qué riesgo asume si el pronóstico se equivoca y conviene adoptar el modelo?**

Se distinguen cuatro pasos que no deben confundirse:
1. **Pronóstico de ventas:** estimación de las ventas semanales.
2. **Política de cantidad:** regla que convierte el pronóstico en una cantidad objetivo semanal (pronóstico + colchón).
3. **Evaluación económica:** costo de los excedentes y faltantes que produce esa política.
4. **Decisión de adopción:** usar, mantener en evaluación o no usar el modelo, según su resultado frente al baseline.

### Diseño del laboratorio
| Elemento | Definición |
|---|---|
| Unidad temporal | Semana de lunes a domingo, identificada por la fecha del lunes |
| Semanas válidas | Solo semanas completas según la cobertura del archivo |
| Variable objetivo | Unidades vendidas (registradas) por semana de un SKU del Reino Unido |
| Test final | Últimas 8 semanas completas; se evalúa una sola vez |
| Selección del SKU | Criterios calculados solo antes del test |
| Validación | Backtesting rolling-origin con horizonte de 8 semanas dentro del entrenamiento |
| Modelos | Naïve, Naïve estacional (m = 4) y Exponential Smoothing |
| Regla de selección (única) | Candidatos con costo promedio ≤ mínimo × 1,01 (incluye exactamente el 1 %); si el mínimo es 0, solo empatan los costos 0. Desempate: menor MAE, menor desviación del costo, menor complejidad |
| Política de colchón | Multiplicador 0; 0,5; 1 o 1,5 × MAE previo; el MAE previo de cada origen usa solo errores anteriores a ese origen |

## Actividad 1 — Revisión de conceptos

Complete la tabla con sus propias palabras e incluya un ejemplo breve. Si su definición permite responder la pregunta de apoyo, es suficiente; si no, corríjala antes de continuar.

| Concepto | Definición con sus propias palabras y ejemplo | Pregunta de apoyo |
|---|---|---|
| Serie temporal | Datos medidos en orden cronológico, donde cada valor depende en parte de los anteriores. Ej.: unidades vendidas del SKU 84077 cada semana. Si se barajan las filas se pierde ese orden y la dependencia entre semanas, que es justo lo que usan los modelos. | ¿Por qué no se pueden barajar sus filas? |
| Semana completa | Semana de lunes a domingo que el archivo cubre en sus siete días. Ej.: la del 2009-12-07. La primera semana del archivo empieza un martes (2009-12-01), así que le falta el lunes y sumaría menos ventas de las reales. | ¿Por qué se excluye la primera semana del archivo? |
| Tendencia y estacionalidad | Tendencia: movimiento sostenido del nivel hacia arriba o hacia abajo. Estacionalidad: patrón que se repite con un periodo fijo (por ejemplo, picos antes de Navidad cada año). No basta con que la serie suba y baje: el patrón tiene que repetirse con regularidad y verse en más de un ciclo. | ¿Basta con que la serie suba y baje para afirmar estacionalidad? |
| Baseline (Naïve y Naïve estacional) | Pronósticos de referencia muy simples. Naïve repite la última semana; Naïve estacional repite lo ocurrido hace m semanas (aquí m = 4). Si un modelo complejo no supera al Naïve, su complejidad no aporta valor y conviene quedarse con la regla simple. | ¿Qué significa que un modelo complejo no supere al Naïve? |
| Exponential Smoothing | Modelo que pondera más las observaciones recientes y puede incluir nivel, tendencia y estacionalidad. Ej.: Holt-Winters aditivo con m = 4. Más componentes no garantizan un mejor pronóstico: pueden ajustarse al ruido y proyectar patrones que no se repiten. | ¿Más componentes garantizan un mejor pronóstico? |
| Partición temporal (train, validación, test) | Se divide la serie respetando el orden: train para ajustar, validación para elegir modelo y política, y test al final para medir una sola vez. El test se mira una sola vez porque, si se usa para corregir decisiones, deja de ser una prueba independiente y el resultado sale optimista. | ¿Por qué el test se mira una sola vez? |
| Backtesting rolling-origin | Se repite el pronóstico desde varios orígenes; cada vez se entrena con todo lo anterior y se evalúa el bloque siguiente. Ej.: 5 folds de 8 semanas. Frente a un único holdout, muestra si el resultado se sostiene en distintos periodos y permite medir la variabilidad. | ¿Qué aporta frente a un único holdout? |
| MAE, RMSE y WAPE | MAE: error absoluto promedio, en unidades. RMSE: raíz del error cuadrático medio, castiga más los errores grandes. WAPE: suma de errores absolutos entre la suma de ventas reales, en proporción. El MAE traduce el error directamente a unidades. | ¿Cuál traduce el error directamente a unidades? |
| Costo asimétrico del error | Faltar y sobrar no cuestan lo mismo. Aquí un faltante cuesta £1,50 y un excedente £0,40. Por eso 20 unidades de menos cuestan £30 y 20 de más solo £8. | ¿Por qué dos errores de 20 unidades pueden costar distinto? |
| Pronóstico, política de cantidad y evaluación económica | El pronóstico estima ventas; la política lo convierte en una cantidad objetivo (pronóstico + colchón); la evaluación económica calcula el costo de excedentes y faltantes. La cantidad objetivo no es lo que hay que comprar: falta restar el inventario utilizable y los pedidos que llegarán a tiempo. | ¿La cantidad objetivo es lo que hay que comprar? |

In [ ]:
%pip install -q ucimlrepo statsmodels plotly scikit-learn openpyxl

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sklearn
import statsmodels

from ucimlrepo import fetch_ucirepo
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

# No se ocultan advertencias globalmente: las de cada ajuste se registran y se revisan al final.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
VERSIONES = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
             "statsmodels": statsmodels.__version__, "scikit-learn": sklearn.__version__}
print("Versiones:", VERSIONES)

Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


# 2. Descarga de datos reales y trazabilidad

Se intenta primero la API oficial de UCI; si falla, el archivo ZIP oficial, del que se leen y concatenan sus **dos hojas**; si ambas fallan, se sube manualmente el archivo descargado desde https://archive.ics.uci.edu/dataset/502. La celda registra fuente, hojas, dimensiones, rango de fechas y versiones.

> Que otra copia tenga el mismo número de filas **no demuestra** que sea idéntica. Con otras versiones de las bibliotecas, los ajustes de Exponential Smoothing pueden producir cifras ligeramente distintas.

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

def leer_dos_hojas(fuente_excel):
    hojas = pd.read_excel(fuente_excel, sheet_name=None)  # "Year 2009-2010" y "Year 2010-2011"
    return pd.concat(hojas.values(), ignore_index=True), list(hojas)

df, fuente, hojas_leidas = None, None, None
try:
    retail = fetch_ucirepo(id=502)
    df, fuente, hojas_leidas = retail.data.original.copy(), "API ucimlrepo (UCI ID 502)", "tabla única entregada por la API"
except Exception as exc:
    print("API ucimlrepo no disponible:", type(exc).__name__)

if df is None:
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=120) as respuesta:
            blob = respuesta.read()
        with ZipFile(BytesIO(blob)) as z:
            nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
            df, hojas_leidas = leer_dos_hojas(BytesIO(z.read(nombre)))
        fuente = "ZIP oficial de UCI"
    except Exception as exc:
        print("ZIP oficial no disponible:", type(exc).__name__)

if df is None:
    locales = sorted(f for f in os.listdir(".")
                     if "online" in f.lower() and "retail" in f.lower() and f.lower().endswith((".xlsx", ".csv")))
    if not locales:
        try:
            from google.colab import files
            print("Suba el archivo online_retail_II.xlsx descargado desde UCI.")
            locales = list(files.upload().keys())
        except ImportError:
            raise FileNotFoundError("Coloque online_retail_II.xlsx en la carpeta de trabajo y vuelva a ejecutar.")
    archivo = locales[0]
    if archivo.lower().endswith(".xlsx"):
        df, hojas_leidas = leer_dos_hojas(archivo)
    else:
        df = pd.read_csv(archivo, dtype={"Invoice": str, "InvoiceNo": str, "StockCode": str})
        hojas_leidas = "no aplica (CSV)"
    fuente = f"archivo local: {archivo}"

fechas_raw = pd.to_datetime(df[[c for c in df.columns if "date" in str(c).lower()][0]], errors="coerce")
print("Fuente utilizada:", fuente)
print("Hojas:", hojas_leidas)
print("Dimensión:", df.shape)
print("Rango de fechas:", fechas_raw.min(), "→", fechas_raw.max())
print("Ejecución:", pd.Timestamp.now(tz="UTC").isoformat())
print("Versiones:", VERSIONES)
display(df.head())

API ucimlrepo no disponible: DatasetNotFoundError
Fuente utilizada: ZIP oficial de UCI
Hojas: ['Year 2009-2010', 'Year 2010-2011']
Dimensión: (1067371, 8)
Rango de fechas: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Ejecución: 2026-10-22T01:39:01.915241+00:00
Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_", ""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_", "")
        if key in normalized:
            return normalized[key]
    return None

mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "CustomerID": find_col(df.columns, ["Customer ID", "CustomerID"]),
    "Country": find_col(df.columns, ["Country"]),
}
faltan = [k for k, v in mapping.items() if v is None and k not in ("Description", "CustomerID")]
if faltan:
    raise ValueError(f"No se identificaron columnas obligatorias: {faltan}")
df = df.rename(columns={v: k for k, v in mapping.items() if v is not None})
print(mapping)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'CustomerID': 'Customer ID', 'Country': 'Country'}


# 3. Control de calidad

Los `StockCode` nulos se descartan **antes** de convertir a texto; de lo contrario, la ausencia se convertiría en un producto llamado «nan».

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
print("StockCode nulos descartados:", int(df["StockCode"].isna().sum()))
df = df[df["StockCode"].notna()].copy()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df = df[df["StockCode"] != ""]
cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")

print("Fecha mínima:", df["InvoiceDate"].min(), "·", df["InvoiceDate"].min().day_name())
print("Fecha máxima:", df["InvoiceDate"].max(), "·", df["InvoiceDate"].max().day_name())
print("Cancelaciones:", int(cancel.sum()))
print("Quantity <= 0:", int((df["Quantity"] <= 0).sum()))
print("UnitPrice <= 0:", int((df["UnitPrice"] <= 0).sum()))
print("Fechas nulas:", int(df["InvoiceDate"].isna().sum()))

clean = df.loc[(~cancel) & df["InvoiceDate"].notna() & (df["Quantity"] > 0) &
               (df["UnitPrice"] > 0) & df["Country"].notna()].copy()
uk = clean[clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")].copy()
print("Filas válidas UK:", len(uk), "· SKUs UK:", uk["StockCode"].nunique())

StockCode nulos descartados: 0
Fecha mínima: 2009-12-01 07:45:00 · Tuesday
Fecha máxima: 2011-12-09 12:50:00 · Friday
Cancelaciones: 19494
Quantity <= 0: 22950
UnitPrice <= 0: 6207
Fechas nulas: 0
Filas válidas UK: 958501 · SKUs UK: 4905


### Pregunta 1
¿Por qué no sería correcto sumar directamente `Quantity` sin eliminar cancelaciones y cantidades no positivas?

**Respuesta:**
Porque `Quantity` mezcla ventas con otras operaciones. Las cancelaciones (facturas que empiezan con «C», 19 494 en el archivo) aparecen con cantidades negativas y anulan ventas hechas antes; si se suman, restan unidades en la semana de la anulación, no en la de la venta, y generan caídas artificiales. Las cantidades no positivas (22 950) y los precios no positivos (6 207) corresponden a devoluciones, ajustes, muestras o errores de registro, no a ventas. Como la variable objetivo son las **ventas registradas**, incluir esos movimientos subestimaría la serie y le metería ruido que ningún modelo podría anticipar.

# 4. Semanas completas y partición temporal

Las semanas van de **lunes a domingo** y se identifican por la fecha del lunes. Una semana es **completa** solo si el archivo cubre sus siete días: se excluyen la primera semana si el archivo no empieza en lunes y la última si no termina en domingo. Las **últimas 8 semanas completas** se reservan como test final.

Algunas semanas no tienen **ninguna venta registrada** en los datos filtrados. El cierre de la tienda es una **posible explicación no verificada**: los datos no lo confirman.

In [ ]:
def lunes(fechas):
    fechas = pd.to_datetime(fechas)
    return fechas.dt.normalize() - pd.to_timedelta(fechas.dt.weekday, unit="D")

uk["week"] = lunes(uk["InvoiceDate"])
inicio_archivo, fin_archivo = df["InvoiceDate"].min(), df["InvoiceDate"].max()
lunes_inicio = inicio_archivo.normalize() - pd.Timedelta(days=inicio_archivo.weekday())
lunes_fin = fin_archivo.normalize() - pd.Timedelta(days=fin_archivo.weekday())
primera_completa = lunes_inicio if inicio_archivo.weekday() == 0 else lunes_inicio + pd.Timedelta(weeks=1)
ultima_completa = lunes_fin if fin_archivo.weekday() == 6 else lunes_fin - pd.Timedelta(weeks=1)

semanas = pd.date_range(primera_completa, ultima_completa, freq="W-MON")
TEST_H = 8
semanas_test = semanas[-TEST_H:]
semanas_train = semanas[:-TEST_H]

registros = uk.groupby("week").size().reindex(semanas, fill_value=0)
semanas_sin_registro = registros[registros == 0].index

print("Semanas excluidas por incompletas:", lunes_inicio.date(), "y", lunes_fin.date())
print("Semanas completas:", len(semanas), "·", semanas[0].date(), "→", semanas[-1].date())
print("Entrenamiento:", len(semanas_train), "semanas ·", semanas_train[0].date(), "→", semanas_train[-1].date())
print("Test final:", len(semanas_test), "semanas ·", semanas_test[0].date(), "→", semanas_test[-1].date())
print("Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada):",
      [d.date() for d in semanas_sin_registro])

Semanas excluidas por incompletas: 2009-11-30 y 2011-12-05
Semanas completas: 104 · 2009-12-07 → 2011-11-28
Entrenamiento: 96 semanas · 2009-12-07 → 2011-10-03
Test final: 8 semanas · 2011-10-10 → 2011-11-28
Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada): [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]


### Pregunta 2
¿Por qué una partición aleatoria con `train_test_split` sería incorrecta en este problema?

**Respuesta:**
Porque `train_test_split` reparte las semanas al azar. El modelo terminaría entrenando con semanas posteriores a las que pronostica (por ejemplo, ver noviembre de 2011 para predecir octubre de 2011), lo que es una fuga de información del futuro. Además, rompe la secuencia que necesitan Naïve, Naïve estacional y Exponential Smoothing, que se basan en las semanas inmediatamente anteriores. En la práctica la empresa solo conoce el pasado cuando planifica, así que la evaluación debe imitar eso: entrenar con lo anterior y evaluar lo siguiente. Con una partición aleatoria el error saldría optimista y no representaría el desempeño real.

# 5. Selección reproducible del producto (solo con el entrenamiento)

Criterios, calculados **exclusivamente con las semanas de entrenamiento**:
1. **Periodo con ventas registradas cercano al entrenamiento completo:** primera venta dentro de las **4 primeras** semanas y última venta dentro de las **4 últimas**. El criterio tolera hasta 3 semanas iniciales o finales sin ventas, que luego se completan con cero aunque podrían corresponder a un periodo sin disponibilidad. **No verifica inventario.**
2. **Cobertura:** ventas registradas en al menos el 90 % de las semanas de entrenamiento con algún registro en la tienda.
3. Entre los elegibles, el de **mayor cantidad de unidades**; desempate por más semanas con ventas.

Se comprueba que haya al menos un producto elegible (y al menos dos para el Reto 3).

In [ ]:
uk_train = uk[uk["week"].isin(semanas_train)]
semanas_con_registro_train = [w for w in semanas_train if w not in set(semanas_sin_registro)]

cobertura = (uk_train.groupby(["StockCode", "week"])["Quantity"].sum().reset_index()
             .groupby("StockCode")
             .agg(semanas_con_venta=("week", "nunique"), unidades=("Quantity", "sum"),
                  primera_venta=("week", "min"), ultima_venta=("week", "max"))
             .reset_index())
cobertura["cobertura"] = cobertura["semanas_con_venta"] / len(semanas_con_registro_train)
elegibles = cobertura[
    (cobertura["primera_venta"] <= semanas_train[3]) &      # dentro de las 4 primeras semanas
    (cobertura["ultima_venta"] >= semanas_train[-4]) &      # dentro de las 4 últimas semanas
    (cobertura["cobertura"] >= 0.90)
].sort_values(["unidades", "semanas_con_venta"], ascending=False).reset_index(drop=True)

if len(elegibles) == 0:
    raise ValueError("Ningún SKU cumple los criterios: revise la carga y la limpieza antes de continuar.")
print("SKUs elegibles:", len(elegibles), "· suficientes para el Reto 3:", len(elegibles) >= 2)
SKU = elegibles.loc[0, "StockCode"]
desc = uk.loc[uk["StockCode"] == SKU, "Description"].dropna().astype(str).mode().iloc[0]
print("SKU seleccionado:", SKU, "·", desc)
display(elegibles.head(5))

SKUs elegibles: 389 · suficientes para el Reto 3: True
SKU seleccionado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS


,StockCode,semanas_con_venta,unidades,primera_venta,ultima_venta,cobertura
0,84077,93,88273,2009-12-07,2011-10-03,0.989362
1,85123A,94,80903,2009-12-07,2011-10-03,1.000000
2,85099B,94,79134,2009-12-07,2011-10-03,1.000000
3,21212,94,67878,2009-12-07,2011-10-03,1.000000
4,17003,88,67514,2009-12-07,2011-10-03,0.936170


# 6. Serie semanal de ventas registradas

**Qué representa un cero.** Un cero indica que **no hay ventas registradas** del SKU en esa semana dentro de los datos filtrados. Los datos no permiten distinguir entre:
- **disponibilidad de inventario** con demanda nula;
- **demanda no satisfecha** por falta de inventario (no queda registrada);
- **ausencia de registros** (por ejemplo, semanas sin ninguna venta registrada en la tienda).

Completar con cero es un **supuesto sobre las ventas observadas**: no demuestra demanda cero ni disponibilidad continua. Por eso el laboratorio pronostica **ventas registradas**, que solo aproximan la demanda.

In [ ]:
serie = (uk[uk["StockCode"] == SKU].groupby("week")["Quantity"].sum()
         .reindex(semanas, fill_value=0).astype(float))
serie.name = "unidades"
train = serie.loc[semanas_train]
test = serie.loc[semanas_test]

ceros = serie[serie == 0].index
print("Semanas de la serie:", len(serie), "· semanas sin ventas registradas del SKU:", len(ceros))
print("  coinciden con semanas sin ningún registro en la tienda:", [d.date() for d in ceros if d in set(semanas_sin_registro)])
print("  semanas con registros en la tienda pero sin ventas del SKU:", [d.date() for d in ceros if d not in set(semanas_sin_registro)])
print("Unidades en entrenamiento:", int(train.sum()))

Semanas de la serie: 104 · semanas sin ventas registradas del SKU: 3
  coinciden con semanas sin ningún registro en la tienda: [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]
  semanas con registros en la tienda pero sin ventas del SKU: [datetime.date(2011, 1, 17)]
Unidades en entrenamiento: 88273


# 7. Exploración (solo con el entrenamiento)

La exploración que orienta decisiones usa únicamente el entrenamiento: mirar el test antes de elegir el modelo sería una fuga de información. En este laboratorio las medias móviles de las **últimas 4 y 8 semanas** se usan para **explorar** el nivel; no se evalúan como método de pronóstico (podrían usarse como tal, pero no forman parte de esta comparación).

In [ ]:
explora = pd.DataFrame({"ventas": train})
explora["media_movil_4"] = explora["ventas"].rolling(4).mean()
explora["media_movil_8"] = explora["ventas"].rolling(8).mean()
fig = go.Figure()
for col, nombre in [("ventas", "Ventas registradas"), ("media_movil_4", "Media móvil 4"), ("media_movil_8", "Media móvil 8")]:
    fig.add_trace(go.Scatter(x=explora.index, y=explora[col], name=nombre))
fig.update_layout(title=f"Entrenamiento · SKU {SKU} · {desc}", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

perfil = (explora.assign(semana_anio=explora.index.isocalendar().week.astype(int))
          .groupby("semana_anio")["ventas"].mean().reset_index())
fig = px.line(perfil, x="semana_anio", y="ventas", markers=True,
              title="Perfil promedio por semana del año (entrenamiento)",
              labels={"semana_anio": "Semana del año", "ventas": "Ventas promedio"})
fig.show()

anual = train.groupby(train.index.year).agg(["count", "mean", "median", "max"])
display(anual.round(1))

,count,mean,median,max
2009,4,493.0,384.0,1204.0
2010,52,1034.4,608.0,7063.0
2011,40,812.8,558.0,3216.0


### Pregunta 3
Identifique evidencia de tendencia, estacionalidad o quiebres en el entrenamiento. Diferencie lo que observa de lo que todavía necesita comprobar, y explique qué representa un cero en la serie.

**Respuesta:**
**Lo que se observa:**
- **Tendencia:** no hay una tendencia clara. La media semanal baja de 1 034,4 unidades en 2010 a 812,8 en 2011 (mediana de 608 a 558), pero el entrenamiento de 2011 termina en octubre y no incluye el otoño, que en 2010 concentró las semanas más altas. La comparación está sesgada y no prueba una caída real del nivel.
- **Estacionalidad:** los picos más grandes se concentran antes de Navidad de 2010: 3 504 unidades (semana del 2010-10-04), 5 301 (2010-11-01) y 7 063 (2010-11-29). La media móvil de 4 semanas llega a cerca de 2 745 en ese periodo, y el perfil por semana del año muestra picos en las semanas 44 y 48. Esto **sugiere** una temporada navideña, pero el entrenamiento solo tiene un otoño completo, así que todavía no se puede afirmar estacionalidad anual. El ciclo de 4 semanas del modelo es un supuesto, no algo que se vea en el gráfico.
- **Quiebres y picos:** hay picos aislados fuera de temporada (2 592 en marzo de 2010, 2 938 en agosto de 2010, 3 216 en marzo de 2011), típicos de pedidos grandes de clientes mayoristas. La media del entrenamiento (≈ 920) queda muy por encima de la mediana (584,5), así que la demanda es irregular y asimétrica. Después de Navidad de 2010 el nivel cae fuerte (media móvil de 4 semanas entre 200 y 340 en enero de 2011) y se recupera entre abril y mayo de 2011.

**Lo que falta comprobar:** si los picos son estacionales o pedidos puntuales de pocos clientes (revisando las facturas), si el menor nivel de 2011 es un cambio persistente y si el patrón de otoño se repite en 2011.

**Qué representa un cero:** la serie tiene 3 semanas en cero. Dos (2009-12-28 y 2010-12-27) coinciden con semanas sin ningún registro en la tienda, posiblemente por cierre navideño (no verificado). La tercera (2011-01-17) tuvo actividad en la tienda pero ninguna venta del SKU. Un cero significa **ausencia de ventas registradas**, no necesariamente demanda cero: también puede haber sido falta de inventario o simplemente que no hubo registros.

# 8. Modelos, cantidad operativa, costo y regla de selección

- **Naïve:** repite la última semana observada.
- **Naïve estacional (m = 4):** repite la semana de hace 4 semanas. El periodo de 4 semanas es un **supuesto de modelado** (ciclo operativo corto), no una propiedad comprobada del negocio; con 96 semanas de entrenamiento no hay suficientes años para estimar con confianza un ciclo anual de 52 semanas.
- **Exponential Smoothing:** nivel y tendencia aditivos con estacionalidad aditiva m = 4. Si un entrenamiento tuviera menos de 2 × 4 + 8 = 16 semanas, se simplifica a nivel y tendencia (documentado; se informa si ocurre).

**Cantidad operativa:** todo pronóstico se convierte en `max(redondeo(pronóstico), 0)` y **todas** las métricas y costos se calculan sobre esa cantidad.

**Costo base (supuesto académico, no proviene de UCI):** £0,40 por unidad excedente y £1,50 por unidad faltante.

**Regla única de selección** (se usa para el modelo, la política de colchón, el Reto 1 y el Reto 3):
1. Candidatos: costo promedio de validación ≤ mínimo × 1,01 (**incluye** exactamente el 1 %; se admite una holgura de 1e-9 por redondeo numérico). Si el mínimo es 0, solo son candidatos los costos iguales a 0.
2. Desempate entre candidatos: menor MAE promedio, luego menor desviación del costo y, al final, menor complejidad.

In [ ]:
SEASON = 4
OVER_COST, SHORT_COST = 0.40, 1.50
MODELOS = ["Naive", "SeasonalNaive", "ExpSmoothing"]   # en orden de complejidad
TOLERANCIA = 0.01
registro_ajustes = []

def ajustar_es(y, etiqueta):
    """Ajusta Exponential Smoothing y registra todas las advertencias del ajuste."""
    if len(y) >= 2 * SEASON + 8:
        modelo = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=SEASON,
                                      initialization_method="estimated")
        especificacion = "nivel + tendencia + estacionalidad aditivas (m = 4)"
    else:
        modelo = ExponentialSmoothing(y, trend="add", seasonal=None, initialization_method="estimated")
        especificacion = "simplificado: nivel + tendencia"
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        ajuste = modelo.fit(optimized=True)
    registro_ajustes.append({"etapa": etiqueta.split(" · ")[0], "ajuste": etiqueta, "n_train": len(y),
                             "especificacion": especificacion, "advertencias": len(avisos),
                             "tipos": "; ".join(sorted({w.category.__name__ for w in avisos})),
                             "mensajes": " | ".join(sorted({str(w.message)[:110] for w in avisos}))})
    return ajuste

def pronosticar(modelo, y, h, etiqueta=""):
    if modelo == "Naive":
        f = np.repeat(y.iloc[-1], h)
    elif modelo == "SeasonalNaive":
        f = np.resize(y.iloc[-SEASON:].values, h)
    elif modelo == "ExpSmoothing":
        f = np.asarray(ajustar_es(y, etiqueta).forecast(h), dtype=float)
    else:
        raise ValueError(modelo)
    return np.maximum(np.round(f), 0)   # cantidad operativa

def evaluar(real, cantidad, over=OVER_COST, short=SHORT_COST):
    real, cantidad = np.asarray(real, float), np.asarray(cantidad, float)
    exceso, faltante = np.maximum(cantidad - real, 0), np.maximum(real - cantidad, 0)
    return {"MAE": mean_absolute_error(real, cantidad),
            "RMSE": mean_squared_error(real, cantidad) ** 0.5,
            "WAPE": np.abs(real - cantidad).sum() / real.sum() if real.sum() > 0 else np.nan,  # no definido si la demanda total es 0
            "unidades_excedentes": exceso.sum(), "unidades_faltantes": faltante.sum(),
            "costo": (exceso * over + faltante * short).sum()}

def resumir(filas, por, orden):
    """Resume los folds por modelo o por política. 'orden' va de la opción más simple a la más compleja."""
    t = (pd.DataFrame(filas).groupby(por)
           .agg(costo_prom=("costo", "mean"), costo_sd=("costo", "std"),
                MAE_prom=("MAE", "mean"), MAE_sd=("MAE", "std"), RMSE_prom=("RMSE", "mean"),
                WAPE_prom=("WAPE", "mean"), excedentes=("unidades_excedentes", "sum"),
                faltantes=("unidades_faltantes", "sum"))
           .reindex(orden))
    t["complejidad"] = range(len(orden))
    return t

def seleccionar(tabla, tol=TOLERANCIA):
    """Regla única de selección (ver el texto de esta sección)."""
    minimo = tabla["costo_prom"].min()
    limite = minimo * (1 + tol)          # si minimo == 0, limite == 0: solo empatan costos 0
    candidatos = tabla[tabla["costo_prom"] <= limite + 1e-9]
    return candidatos.sort_values(["MAE_prom", "costo_sd", "complejidad"], kind="stable").index[0]
print("Funciones listas.")

Funciones listas.


# 9. Backtesting rolling-origin (solo con el entrenamiento)

Cada fold entrena con todo lo anterior a su origen y pronostica las **8 semanas** siguientes, igual que el test final. Se exige al menos 52 semanas de entrenamiento; los bloques de validación no se superponen y el último termina justo antes del test. La celda **comprueba** que ningún fold invada el test.

In [ ]:
BT_H, MIN_TRAIN = 8, 52
origenes = list(range(len(train) - BT_H, MIN_TRAIN - 1, -BT_H))[::-1]

def backtest(y, modelos, etiqueta):
    """Pronostica cada bloque de validación desde su origen. Devuelve métricas por fold y cantidades."""
    filas, cantidades = [], {}
    for k, origen in enumerate(origenes, start=1):
        tr, va = y.iloc[:origen], y.iloc[origen:origen + BT_H]
        assert tr.index[-1] < va.index[0] and va.index[-1] < semanas_test[0], "un fold invade el test"
        for m in modelos:
            cantidades[(k, m)] = pronosticar(m, tr, BT_H, f"{etiqueta} · fold {k}")
            filas.append({**evaluar(va, cantidades[(k, m)]), "fold": k, "modelo": m})
    return filas, cantidades

filas_bt, q_bt = backtest(train, MODELOS, "backtesting")
bt = pd.DataFrame(filas_bt)
folds = pd.DataFrame([{"fold": k, "entrenamiento": f"{train.index[0].date()} → {train.index[o - 1].date()}", "n_train": o,
                       "validacion": f"{train.index[o].date()} → {train.index[o + BT_H - 1].date()}"}
                      for k, o in enumerate(origenes, start=1)])
display(folds)
display(bt.round(2))
print("Comprobado: los", len(origenes), "folds terminan antes del test (", semanas_test[0].date(), ").")

,fold,entrenamiento,n_train,validacion
0,1,2009-12-07 → 2010-12-27,56,2011-01-03 → 2011-02-21
1,2,2009-12-07 → 2011-02-21,64,2011-02-28 → 2011-04-18
2,3,2009-12-07 → 2011-04-18,72,2011-04-25 → 2011-06-13
3,4,2009-12-07 → 2011-06-13,80,2011-06-20 → 2011-08-08
4,5,2009-12-07 → 2011-08-08,88,2011-08-15 → 2011-10-03


,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo,fold,modelo
0,608.12,1044.62,1.00,0.0,4865.0,7297.5,1,Naive
1,528.88,758.45,0.87,1195.0,3036.0,5032.0,1,SeasonalNaive
2,1599.12,1838.12,2.63,11475.0,1318.0,6567.0,1,ExpSmoothing
3,1111.12,1460.84,0.83,113.0,8776.0,13209.2,2,Naive
4,789.88,1000.51,0.59,1317.0,5002.0,8029.8,2,SeasonalNaive
5,949.12,1147.68,0.71,4251.0,3342.0,6713.4,2,ExpSmoothing
6,723.88,824.92,0.77,4550.0,1241.0,3681.5,3,Naive
7,1248.88,1463.50,1.33,8557.0,1434.0,5573.8,3,SeasonalNaive
8,850.75,960.03,0.91,5384.0,1422.0,4286.6,3,ExpSmoothing
9,391.88,471.10,0.75,47.0,3088.0,4650.8,4,Naive


Comprobado: los 5 folds terminan antes del test ( 2011-10-10 ).


In [ ]:
resumen_val = resumir(filas_bt, "modelo", MODELOS)
SELECCIONADO = seleccionar(resumen_val)
display(resumen_val.round(2))
print("MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test):", SELECCIONADO)
fig = px.box(bt, x="modelo", y="costo", points="all", title="Costo simulado por fold de validación (£)",
             labels={"modelo": "Modelo", "costo": "Costo por fold (£)"})
fig.show()

,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,6239.22,4296.41,620.2,327.68,819.13,0.75,5469.0,19339.0,0
SeasonalNaive,4950.34,2065.83,666.5,370.02,830.61,0.83,13853.0,12807.0,1
ExpSmoothing,4324.10,2335.11,927.3,420.72,1063.63,1.29,30925.0,6167.0,2


MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test): ExpSmoothing


### Pregunta 4
¿Qué modelo se seleccionó en validación y con qué regla? ¿Coinciden el ranking por costo y el ranking por MAE o RMSE? ¿Por qué el ranking por WAPE no aporta un criterio adicional al MAE?

**Respuesta:**
Se seleccionó **Exponential Smoothing** con la regla única de selección. Su costo promedio de validación fue £4 324,10, el mínimo. El límite de candidatos fue 4 324,10 × 1,01 = £4 367,34; Naïve estacional (£4 950,34) y Naïve (£6 239,22) quedaron fuera, así que hubo un solo candidato y no hizo falta desempatar.

Los rankings **no coinciden**, de hecho se invierten:
- Por costo: Exponential Smoothing (£4 324,10) < Naïve estacional (£4 950,34) < Naïve (£6 239,22).
- Por MAE: Naïve (620,2) < Naïve estacional (666,5) < Exponential Smoothing (927,3).
- Por RMSE: Naïve (819,13) < Naïve estacional (830,61) < Exponential Smoothing (1 063,63).

La razón es la asimetría del costo. Exponential Smoothing se equivoca sobre todo por exceso (30 925 unidades excedentes frente a 6 167 faltantes), mientras que Naïve se equivoca por defecto (19 339 faltantes). Como cada faltante cuesta 3,75 veces más que un excedente, el modelo que más se equivoca en unidades resulta el más barato.

El WAPE no aporta un criterio adicional porque es la suma de errores absolutos dividida entre la suma de ventas reales. En cada fold ese denominador es el mismo para los tres modelos, así que el WAPE es el MAE reescalado por una constante y ordena los modelos de la misma forma (0,75; 0,83; 1,29). Además, igual que el MAE, no distingue si el error fue por exceso o por defecto, que es lo que define el costo.

### Pregunta 5
¿Qué tan estable fue cada modelo entre folds? Use la desviación del costo y del MAE.

**Respuesta:**
- **Naïve** fue el menos estable en costo: desviación de £4 296,41 sobre una media de £6 239,22 (coeficiente de variación ≈ 0,69), con folds entre £2 357,10 y £13 209,20. Su MAE fue el más estable (desviación 327,68), pero cuando falla lo hace por faltantes, y eso se paga caro.
- **Naïve estacional** fue el más estable en costo: desviación de £2 065,83 (CV ≈ 0,42), con folds entre £2 934,30 y £8 029,80. Su desviación del MAE fue 370,02.
- **Exponential Smoothing** quedó en un punto intermedio: desviación de £2 335,11 (CV ≈ 0,54), con folds entre £1 519,50 y £6 713,40, y la mayor desviación del MAE (420,72). En el fold 1 tuvo un MAE de 1 599,12 porque, después del pico navideño, sobreestimó (11 475 unidades excedentes).

Exponential Smoothing costó menos que Naïve en 4 de los 5 folds (solo perdió el fold 3: £4 286,60 frente a £3 681,50). Aun así, la variabilidad es alta en todos los modelos: el costo cambia mucho según el periodo, así que una diferencia promedio de validación no garantiza el mismo orden en un periodo nuevo.

# 10. Política de colchón calibrada sin información futura (antes del test)

El colchón suma unidades al pronóstico para protegerse de faltantes: **colchón = ⌈multiplicador × MAE previo⌉**. Se comparan los multiplicadores 0; 0,5; 1 y 1,5.

**Regla de calibración (sin errores futuros).** Para cada origen, el *MAE previo* se calcula con el modelo seleccionado en los **2 bloques de 8 semanas inmediatamente anteriores** a ese origen: cada bloque se pronostica entrenando solo con lo anterior a él. Así, el colchón del fold 1 usa solo errores anteriores al fold 1.
- Si no hay historia suficiente (algún bloque quedaría con menos de 16 semanas de entrenamiento), el colchón de ese origen es **0** y se informa. Con 52 semanas mínimas no ocurre en este laboratorio.
- La política (modelo, multiplicador y regla) se elige con la **regla única de selección** y queda **fijada antes del test**.
- Para el test, el MAE previo se calcula con la misma regla desde el final del entrenamiento (solo información previa al test).
- Después de evaluar, la **magnitud** del colchón se actualiza con la historia completa para proyectar, **sin cambiar la regla**.

In [ ]:
MULTIPLICADORES = [0, 0.5, 1.0, 1.5]
BLOQUES_CAL = 2

def mae_previo(modelo, y, etiqueta, bloques=BLOQUES_CAL):
    """MAE del modelo en los 'bloques' bloques de 8 semanas que terminan al final de y.
    Solo usa y (información hasta el origen). Devuelve None si la historia no alcanza."""
    errores = []
    for b in range(bloques, 0, -1):
        corte = len(y) - b * BT_H
        if corte < 2 * SEASON + 8:
            return None
        real = y.iloc[corte:corte + BT_H].values
        errores.append(np.abs(real - pronosticar(modelo, y.iloc[:corte], BT_H, f"{etiqueta} · bloque {b}")))
    return float(np.mean(np.concatenate(errores)))

def colchon(mult, mae):
    return 0 if (mae is None or mult == 0) else int(np.ceil(mult * mae))

def comparar_politicas(bloques=BLOQUES_CAL, etiqueta="calibración"):
    filas = []
    for k, origen in enumerate(origenes, start=1):
        tr, va = train.iloc[:origen], train.iloc[origen:origen + BT_H]
        mae_k = mae_previo(SELECCIONADO, tr, f"{etiqueta} · fold {k}", bloques)
        for mult in MULTIPLICADORES:
            c = colchon(mult, mae_k)
            filas.append({**evaluar(va, q_bt[(k, SELECCIONADO)] + c), "fold": k, "multiplicador": mult,
                          "mae_previo": mae_k, "colchon": c,
                          "ultima_semana_calibracion": tr.index[-1].date(), "inicio_validacion": va.index[0].date()})
    return pd.DataFrame(filas)

pol_folds = comparar_politicas()
assert (pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all(), "colchón con información futura"
resumen_pol = resumir(pol_folds, "multiplicador", MULTIPLICADORES)
MULT = seleccionar(resumen_pol)

display(pol_folds.loc[pol_folds["multiplicador"] == 1.0,
                      ["fold", "ultima_semana_calibracion", "inicio_validacion", "mae_previo"]].round(1))
display(resumen_pol.round(2))
MAE_TEST = mae_previo(SELECCIONADO, train, "calibración test")       # solo entrenamiento
COLCHON_TEST = colchon(MULT, MAE_TEST)
POLITICA = f"{SELECCIONADO} + {MULT:g} × MAE previo"
print("POLÍTICA FIJADA ANTES DEL TEST:", POLITICA)
print(f"Colchón para el test: {COLCHON_TEST} unidades por semana (MAE previo = {MAE_TEST:.1f}, calculado solo con el entrenamiento)")

,fold,ultima_semana_calibracion,inicio_validacion,mae_previo
2,1,2010-12-27,2011-01-03,1400.7
6,2,2011-02-21,2011-02-28,1533.2
10,3,2011-04-18,2011-04-25,1274.1
14,4,2011-06-13,2011-06-20,899.9
18,5,2011-08-08,2011-08-15,821.3


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1
1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2
1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3


POLÍTICA FIJADA ANTES DEL TEST: ExpSmoothing + 0 × MAE previo
Colchón para el test: 0 unidades por semana (MAE previo = 618.8, calculado solo con el entrenamiento)


### Pregunta 6
¿Qué política de colchón quedó fijada antes del test y por qué? Explique por qué el colchón de cada fold se calcula solo con errores anteriores a su origen.

**Respuesta:**
Quedó fijada la política **ExpSmoothing + 0 × MAE previo**, es decir, **sin colchón** (colchón del test = 0 unidades, con un MAE previo de 618,8 calculado solo con el entrenamiento). Fue la de menor costo promedio de validación (£4 324,10) y la única candidata: 0,5× costó £4 806,72, 1× £5 833,74 y 1,5× £7 674,08, todas por encima del límite de £4 367,34.

El colchón sí reduce faltantes (6 167 → 2 440 → 149 → 0), pero aumenta mucho más los excedentes (30 925 → 50 934 → 72 363 → 95 926). Como Exponential Smoothing ya tiende a sobreestimar, sumar unidades significa pagar £0,40 por cada excedente en la mayoría de las semanas a cambio de evitar pocos faltantes.

El colchón de cada fold se calcula solo con errores anteriores a su origen porque así se planifica en la realidad: cuando la empresa decide la cantidad de una semana, solo conoce los errores del pasado. Si se usaran los errores del mismo bloque que se evalúa, el colchón quedaría «a la medida» de un error que todavía no ocurrió y la comparación sería optimista (fuga de información). La verificación de la celda confirma que la última semana de calibración siempre es anterior al inicio de la validación.

# 11. Evaluación final en test (una sola vez)

Se reentrena cada modelo con las 96 semanas de entrenamiento y se pronostican las 8 semanas de test. Se evalúan los tres modelos sin colchón y la **política fijada**. Si otra opción resulta mejor aquí, se informa, pero **no** cambia la selección.

In [ ]:
cantidades_test = {m: pronosticar(m, train, TEST_H, "test") for m in MODELOS}
q_politica = cantidades_test[SELECCIONADO] + COLCHON_TEST
filas = [{**evaluar(test, q), "opcion": f"{m} (sin colchón)"} for m, q in cantidades_test.items()]
if MULT > 0:
    filas.append({**evaluar(test, q_politica), "opcion": f"Política fijada: {POLITICA}"})
res_test = pd.DataFrame(filas).set_index("opcion")
display(res_test.round(2))

fila_pol = f"Política fijada: {POLITICA}" if MULT > 0 else f"{SELECCIONADO} (sin colchón)"
costo_pol, costo_naive = res_test.loc[fila_pol, "costo"], res_test.loc["Naive (sin colchón)", "costo"]
diferencia = costo_naive - costo_pol        # > 0 ahorro; < 0 sobrecosto
print("Política evaluada:", POLITICA)
if SELECCIONADO == "Naive" and MULT == 0:
    print("La política seleccionada es el propio Naïve: no hay diferencia que calcular frente a él.")
elif costo_naive == 0:
    print(f"Diferencia frente a Naïve: £{diferencia:.2f}. No se calcula porcentaje porque el costo de Naïve es cero.")
elif diferencia > 0:
    print(f"Ahorro frente a Naïve: £{diferencia:.2f} ({diferencia / costo_naive * 100:.1f} %).")
else:
    print(f"Sobrecosto frente a Naïve: £{-diferencia:.2f} ({-diferencia / costo_naive * 100:.1f} %).")
mejor_retro = res_test["costo"].idxmin()
if mejor_retro != fila_pol:
    print(f"Retrospectivamente, «{mejor_retro}» habría costado menos en test; la selección no cambia.")

fig = go.Figure()
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode="lines+markers", name="Ventas reales", line=dict(width=3)))
for m, q in cantidades_test.items():
    fig.add_trace(go.Scatter(x=test.index, y=q, mode="lines+markers", name=m))
fig.update_layout(title="Test final: ventas reales y cantidades operativas", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
opcion,,,,,,
Naive (sin colchón),678.75,1545.12,0.48,961.0,4469.0,7087.9
SeasonalNaive (sin colchón),773.75,1631.81,0.54,641.0,5549.0,8579.9
ExpSmoothing (sin colchón),776.62,1700.86,0.54,329.0,5884.0,8957.6


Política evaluada: ExpSmoothing + 0 × MAE previo
Sobrecosto frente a Naïve: £1869.70 (26.4 %).
Retrospectivamente, «Naive (sin colchón)» habría costado menos en test; la selección no cambia.


### Pregunta 7
¿La política seleccionada cumplió en test? Compare su costo con los baselines e indique el ahorro o sobrecosto frente a Naïve. ¿El modelo con menor RMSE fue también el de menor costo?

**Respuesta:**
**No cumplió.** En el test, la política seleccionada (Exponential Smoothing sin colchón) fue la opción **más cara**: £8 957,60, frente a £8 579,90 de Naïve estacional y £7 087,90 de Naïve. Frente a Naïve tuvo un **sobrecosto de £1 869,70 (26,4 %)**. También tuvo el peor MAE (776,62 frente a 678,75 de Naïve).

El modelo con menor RMSE en el test fue **Naïve** (1 545,12), que además fue el de menor costo, así que aquí ambos criterios coinciden. Eso contrasta con la validación, donde el ranking por costo era el inverso del ranking por RMSE: el orden de los modelos cambió de un periodo a otro.

### Diagnóstico del pico del test (retrospectivo)

Se identifica la semana de mayor venta del test y el pedido más grande de esa semana. Un pedido extraordinario **aislado** es un **pico puntual**; un **quiebre estructural** exige un cambio persistente de nivel o patrón, que 8 semanas no permiten confirmar. Los modelos usados son **univariantes**: solo ven las ventas pasadas del SKU y no disponían de información específica sobre ese pedido.

La evaluación principal **conserva** esa semana. El cálculo sin ella es un **análisis retrospectivo de sensibilidad**, no una evaluación alternativa.

In [ ]:
semana_pico = test.idxmax()
lineas_pico = uk[(uk["StockCode"] == SKU) & (uk["week"] == semana_pico)]
pedidos = lineas_pico.groupby(["InvoiceNo", "CustomerID"], dropna=False)["Quantity"].sum().sort_values(ascending=False)
print(f"Semana de mayor venta en test: {semana_pico.date()} · {test.max():.0f} unidades")
factura, cliente = pedidos.index[0]
cliente = "sin dato" if pd.isna(cliente) else str(cliente).removesuffix(".0")
print(f"Pedido más grande: factura {factura} · cliente {cliente} · {pedidos.iloc[0]:.0f} unidades "
      f"({pedidos.iloc[0] / test.max() * 100:.1f} % de la semana)")
print("Mediana semanal del entrenamiento:", train.median(), "unidades")

sin_pico = test.index != semana_pico
sens = pd.DataFrame({m: evaluar(test[sin_pico], cantidades_test[m][sin_pico]) for m in MODELOS}).T
if MULT > 0:
    sens.loc["Política fijada"] = evaluar(test[sin_pico], q_politica[sin_pico])
print("\nANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.")
display(sens[["MAE", "unidades_excedentes", "unidades_faltantes", "costo"]].round(2))

Semana de mayor venta en test: 2011-10-24 · 5328 unidades
Pedido más grande: factura 573008 · cliente 12901 · 4800 unidades (90.1 % de la semana)
Mediana semanal del entrenamiento: 584.5 unidades

ANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.


,MAE,unidades_excedentes,unidades_faltantes,costo
Naive,155.86,961.0,130.0,579.4
SeasonalNaive,232.86,641.0,989.0,1739.9
ExpSmoothing,205.86,329.0,1112.0,1799.6


### Pregunta 8
¿Qué error fue más grave en el test: sobreestimar o subestimar? Use las unidades y los costos. ¿El pico del test indica un quiebre estructural o un pico puntual? ¿Qué muestra, y qué no, el análisis de sensibilidad sin esa semana?

**Respuesta:**
**Fue más grave subestimar.** Exponential Smoothing dejó 5 884 unidades faltantes (5 884 × £1,50 = £8 826,00) y solo 329 excedentes (329 × £0,40 = £131,60): el 98,5 % del costo vino de los faltantes. Lo mismo pasó con Naïve (4 469 faltantes = £6 703,50 frente a 961 excedentes = £384,40).

**El pico es puntual.** La semana del 2011-10-24 vendió 5 328 unidades, unas 9 veces la mediana semanal del entrenamiento (584,5). De ellas, 4 800 (90,1 %) corresponden a un solo pedido: factura 573008 del cliente 12901. Sin ese pedido, la semana habría tenido unas 528 unidades, un valor normal. Además, las semanas siguientes vuelven a niveles de 834 a 1 119 unidades. Un pedido extraordinario aislado no es un cambio persistente de nivel, y con 8 semanas tampoco se podría confirmar un quiebre estructural. Solo esa semana explica 4 772 de los 5 884 faltantes de Exponential Smoothing.

**Lo que muestra el análisis de sensibilidad:** sin la semana del pico, Naïve sigue siendo el más barato (£579,40), frente a Naïve estacional (£1 739,90) y Exponential Smoothing (£1 799,60). O sea, el mal resultado de Exponential Smoothing no se debe solo al pico: también subestimó el resto del test (1 112 faltantes frente a 130 de Naïve), porque su ciclo de 4 semanas proyectó valles de 556 y 561 unidades cuando las ventas se mantuvieron alrededor de 830 a 1 120.

**Lo que no muestra:** es un análisis retrospectivo, hecho después de ver el test, así que no sirve para volver a elegir modelo ni para decir que un modelo «habría ganado». Usa solo 7 semanas, y quitar el pico esconde un riesgo real del negocio: los pedidos grandes de clientes mayoristas.

# 12. Reentrenamiento final y proyección académica

Después de cerrar la evaluación se reentrena el modelo seleccionado con **toda** la historia (104 semanas) y se actualiza la **magnitud** del colchón con la misma regla (los 2 bloques más recientes, que ahora incluyen el test). Se muestra también la proyección de Naïve como referencia.

La tabla es una **proyección académica**, no una orden de compra:
- La **cantidad objetivo semanal** es un objetivo para cada semana. El laboratorio **no simula el arrastre de inventario** entre semanas: si una semana sobra mercadería, esta reduce la necesidad de la siguiente. Por eso la suma de los objetivos semanales no equivale a la cantidad a comprar.
- Una compra simplificada para una semana sería `max(objetivo − inventario utilizable − pedidos que llegarán a tiempo, 0)`: nunca negativa, y solo cuenta el inventario apto para la venta y los pedidos que llegarán antes de esa semana. El laboratorio no dispone de esos datos.

In [ ]:
FINAL_H = 8
pron_final = pronosticar(SELECCIONADO, serie, FINAL_H, "final")
pron_naive = pronosticar("Naive", serie, FINAL_H, "final")
MAE_FINAL = mae_previo(SELECCIONADO, serie, "calibración final")   # misma regla, historia completa
COLCHON_FINAL = colchon(MULT, MAE_FINAL)
semanas_futuras = pd.date_range(serie.index[-1] + pd.Timedelta(weeks=1), periods=FINAL_H, freq="W-MON")
plan = pd.DataFrame({"Semana": semanas_futuras.date,
                     f"Pronóstico {SELECCIONADO}": pron_final.astype(int),
                     "Colchón (política fijada)": COLCHON_FINAL,
                     "Cantidad objetivo semanal": (pron_final + COLCHON_FINAL).astype(int),
                     "Referencia Naïve": pron_naive.astype(int)})
display(plan)
print(f"Política aplicada: {POLITICA} · MAE previo actualizado = {MAE_FINAL:.1f} · colchón = {COLCHON_FINAL} unidades por semana")
print("Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.")

,Semana,Pronóstico ExpSmoothing,Colchón (política fijada),Cantidad objetivo semanal,Referencia Naïve
0,2011-12-05,1075,0,1075,689
1,2011-12-12,899,0,899,689
2,2011-12-19,1617,0,1617,689
3,2011-12-26,998,0,998,689
4,2012-01-02,1103,0,1103,689
5,2012-01-09,927,0,927,689
6,2012-01-16,1645,0,1645,689
7,2012-01-23,1027,0,1027,689


Política aplicada: ExpSmoothing + 0 × MAE previo · MAE previo actualizado = 611.1 · colchón = 0 unidades por semana
Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.


# 13. Decisión de adopción y resumen automático

La decisión se toma con una regla fijada de antemano:
- **Adoptar** (como piloto controlado): la política seleccionada costó menos que Naïve en el test **y** en la mayoría de los folds de validación.
- **Mantener en evaluación:** superó a Naïve en el test, pero no de forma consistente en validación.
- **No adoptar como sustituto de Naïve:** en el test costó igual o más que Naïve. Naïve sigue como referencia; el modelo puede seguir evaluándose sin decidir compras con él.

Un **piloto** sirve para comprobar, durante varias semanas nuevas y con la misma regla, si la política reduce el costo frente a Naïve; no justifica por sí mismo una cantidad de compra.

In [ ]:
naive_bt = bt[bt["modelo"] == "Naive"].set_index("fold")["costo"]
pol_bt = pol_folds[pol_folds["multiplicador"] == MULT].set_index("fold")["costo"]
folds_ganados = int((pol_bt < naive_bt).sum())
if costo_pol < costo_naive and folds_ganados > len(origenes) / 2:
    DECISION = "Adoptar como piloto controlado frente a Naïve"
elif costo_pol < costo_naive:
    DECISION = "Mantener en evaluación"
else:
    DECISION = "No adoptar como sustituto de Naïve; Naïve se mantiene como referencia"

resumen = {
    "Producto analizado": f"{SKU} · {desc}",
    "Seleccionado antes del test": f"{SELECCIONADO}; política {POLITICA} (regla única de selección)",
    "Costo promedio de validación de la política": f"£{resumen_pol.loc[MULT, 'costo_prom']:.2f} ± £{resumen_pol.loc[MULT, 'costo_sd']:.2f} ({len(origenes)} folds)",
    "Folds de validación en que superó a Naïve": f"{folds_ganados} de {len(origenes)}",
    "Resultado en test (8 semanas)": f"£{costo_pol:.2f} frente a £{costo_naive:.2f} de Naïve",
    "Frente a Naïve": ("ahorro" if diferencia > 0 else "sobrecosto") + f" de £{abs(diferencia):.2f}" + (f" ({abs(diferencia) / costo_naive * 100:.1f} %)" if costo_naive > 0 else ""),
    "Unidades excedentes / faltantes en test": f"{res_test.loc[fila_pol, 'unidades_excedentes']:.0f} / {res_test.loc[fila_pol, 'unidades_faltantes']:.0f}",
    "Decisión": DECISION,
    "Proyección académica (suma de objetivos semanales)": f"{int(plan['Cantidad objetivo semanal'].sum())} unidades; no es una orden de compra",
}
for k, v in resumen.items():
    print(f"{k}: {v}")

Producto analizado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS
Seleccionado antes del test: ExpSmoothing; política ExpSmoothing + 0 × MAE previo (regla única de selección)
Costo promedio de validación de la política: £4324.10 ± £2335.11 (5 folds)
Folds de validación en que superó a Naïve: 4 de 5
Resultado en test (8 semanas): £8957.60 frente a £7087.90 de Naïve
Frente a Naïve: sobrecosto de £1869.70 (26.4 %)
Unidades excedentes / faltantes en test: 329 / 5884
Decisión: No adoptar como sustituto de Naïve; Naïve se mantiene como referencia
Proyección académica (suma de objetivos semanales): 9291 unidades; no es una orden de compra


### Pregunta 9 — Cierre ejecutivo
Cierre ejecutivo (máximo 12 líneas). Distinga: modelo y política seleccionados antes del test; resultado de su evaluación frente a Naïve; decisión (adoptar, mantener en evaluación o no adoptar) y su fundamento; proyección académica y por qué no es una orden de compra; una condición que cambiaría la decisión; y las limitaciones (supuestos de costos, ceros, disponibilidad y un solo test de 8 semanas).

**Respuesta:**
1. **Selección antes del test:** Exponential Smoothing (nivel + tendencia + estacionalidad aditiva, m = 4) con la política sin colchón (0 × MAE previo); costo promedio de validación de £4 324,10 ± £2 335,11 y menor costo que Naïve en 4 de 5 folds.
2. **Resultado en test (8 semanas):** costó £8 957,60 frente a £7 087,90 de Naïve, un sobrecosto de £1 869,70 (26,4 %), con 5 884 unidades faltantes y 329 excedentes.
3. **Decisión: no adoptar como sustituto de Naïve.** La regla exige costar menos que Naïve en el test y no se cumplió; Naïve queda como referencia y el modelo puede seguir evaluándose sin decidir compras con él.
4. **Proyección académica:** 9 291 unidades en 8 semanas (entre 899 y 1 645 por semana). No es una orden de compra: no descuenta inventario utilizable ni pedidos en tránsito, ni el arrastre de excedentes entre semanas.
5. **Qué cambiaría la decisión:** que, en varias semanas nuevas evaluadas con la misma regla, el modelo cueste menos que Naïve de forma consistente, o que el área comercial anticipe los pedidos grandes de clientes mayoristas.
6. **Limitaciones:** los costos (£1,50 y £0,40) son supuestos académicos; los ceros no distinguen demanda nula de falta de inventario; no se conoce la disponibilidad real; y el test es uno solo, de 8 semanas, dominado por un pedido de 4 800 unidades.

# 14. Retos individuales de aplicación

La parte guiada ya desarrolló los procedimientos. En esta sección los aplicará a tres casos empresariales. Cada reto tiene cuatro apartados: **A** plantear o preparar el análisis, **B** implementar y obtener resultados, **C** comparar e interpretar la evidencia y **D** formular una decisión justificada con sus limitaciones. Reutilice las funciones de la parte guiada (`evaluar`, `resumir`, `seleccionar`, `backtest`, `pronosticar`, `mae_previo`, `colchon` y `comparar_politicas`) y los objetos ya calculados (`train`, `test`, `origenes`, `q_bt`, `cantidades_test`, `elegibles`). No modifique datos, semanas, pronósticos ni el test. Las respuestas deben ser breves, pero suficientes para justificar la decisión con cifras.

> **Entrega.** «Resuelva los tres retos y sus apartados A, B, C y D en su notebook personal. Guarde la versión ejecutada, con resultados e interpretaciones, en la carpeta semana10/ de su repositorio personal de GitHub y en su propio Google Colab. En la actividad de la semana, en el espacio habilitado, presente ambos enlaces: GitHub y Colab. Verifique que permitan la revisión del trabajo.»

## Reto 1 — Una campaña cambia el costo de los faltantes

**Caso.** La empresa prepara una campaña comercial. Durante la campaña, quedarse sin producto significa perder ventas y clientes que llegaron por la promoción, mientras que guardar unidades sobrantes cuesta menos. El responsable de abastecimiento necesita saber si la recomendación del modelo cambia cuando el faltante se vuelve más costoso.

**Problema.** ¿Se elegiría el mismo método de pronóstico bajo distintos costos del error y, con la evidencia del test, cuál debería usar abastecimiento como referencia?

**Datos, supuestos y recursos**
- Producto y semanas del laboratorio; pronósticos de validación ya calculados (`q_bt`) y cantidades de test (`cantidades_test`).
- Escenarios de costo (supuestos académicos, no provienen de UCI): A faltante £1,00 y exceso £0,50; B faltante £1,50 y exceso £0,40 (base); C faltante £3,00 y exceso £0,25.
- Regla común de selección (`resumir` y `seleccionar`). No se modifican ventas ni pronósticos para favorecer un resultado.

**Apartados**
- **A.** Plantear los escenarios y la decisión
- **B.** Calcular los costos de validación y seleccionar
- **C.** Evaluar en test y comparar frente a Naïve
- **D.** Recomendar a abastecimiento

**Evidencias que deben quedar en el notebook**
- Tabla de escenarios con la razón faltante / exceso.
- Tabla de costos promedio de validación por escenario y modelo, con el seleccionado.
- Tabla de test con excedentes, faltantes, costo y diferencia frente a Naïve, y su gráfico.
- Interpretaciones A–D.

### Reto 1 · Apartado A — Plantear los escenarios y la decisión
Construya una tabla con los tres escenarios y la razón faltante / exceso. Explique qué representa cada escenario para el negocio y formule la decisión empresarial que se tomará con el análisis.

In [ ]:
ESCENARIOS = {"A": (1.00, 0.50), "B": (1.50, 0.40), "C": (3.00, 0.25)}   # (faltante, exceso) en £ por unidad
# Tabla de escenarios: costo por unidad faltante, por unidad excedente y cuántas veces pesa más el faltante
r1_escenarios = pd.DataFrame([{"escenario": esc, "faltante": short, "exceso": over, "faltante / exceso": short / over}
                              for esc, (short, over) in ESCENARIOS.items()]).set_index("escenario")
display(r1_escenarios)

**Interpretación 1A:**
Los tres escenarios hacen que el faltante pese cada vez más que el excedente: en A pesa 2 veces más (£1,00 frente a £0,50), en B 3,75 veces (el caso base) y en C 12 veces (£3,00 frente a £0,25).
- **A:** operación normal, sin campaña; quedarse corto duele poco más que sobrar.
- **B:** el supuesto base del laboratorio.
- **C:** la campaña: perder una venta también es perder un cliente que llegó por la promoción, mientras que guardar unidades cuesta poco.

**Decisión empresarial:** qué método de pronóstico debe usar abastecimiento como referencia durante la campaña, y si esa elección cambia cuando el faltante se vuelve más caro.

### Reto 1 · Apartado B — Calcular los costos de validación y seleccionar
Para cada escenario, recalcule el costo de cada fold con los pronósticos ya disponibles (`q_bt`), resuma por modelo y aplique la regla común de selección. Presente el costo promedio de validación de los tres modelos y el seleccionado en cada escenario.

In [ ]:
r1_val, r1_eleccion = {}, {}
for esc, (short, over) in ESCENARIOS.items():
    # Mismos pronósticos de validación (q_bt); solo cambian los costos unitarios del escenario
    filas = [{**evaluar(train.iloc[o:o + BT_H], q_bt[(k, m)], over=over, short=short), "fold": k, "modelo": m}
             for k, o in enumerate(origenes, start=1) for m in MODELOS]
    r1_val[esc] = resumir(filas, "modelo", MODELOS)
    r1_eleccion[esc] = seleccionar(r1_val[esc])
# Costo promedio de validación por escenario y modelo, con el modelo que elige la regla común
tabla_r1b = pd.DataFrame({esc: r1_val[esc]["costo_prom"] for esc in ESCENARIOS}).T
tabla_r1b.index.name = "escenario"
tabla_r1b["seleccionado"] = pd.Series(r1_eleccion)
tabla_r1b["candidatos (≤ mínimo × 1,01)"] = [", ".join(r1_val[e].index[r1_val[e]["costo_prom"] <= r1_val[e]["costo_prom"].min() * (1 + TOLERANCIA) + 1e-9])
                                            for e in ESCENARIOS]
display(tabla_r1b.round(2))

**Interpretación 1B:**
La selección **sí cambia** según el escenario. Con los costos de validación de Colab:
- **Escenario A:** gana **Naïve estacional** (£3 946,70), frente a Exponential Smoothing (£4 325,90) y Naïve (£4 414,70).
- **Escenario B:** gana **Exponential Smoothing** (£4 324,10), frente a Naïve estacional (£4 950,34) y Naïve (£6 239,22).
- **Escenario C:** gana **Exponential Smoothing** (£5 246,45), frente a Naïve estacional (£8 376,85) y Naïve (£11 876,85).

En ningún escenario hubo otro modelo dentro del 1 % del mínimo, así que la selección no dependió de la tolerancia. El patrón tiene lógica: Exponential Smoothing es el que menos faltantes genera en validación (6 167, frente a 12 807 de Naïve estacional y 19 339 de Naïve), así que, mientras más caro es el faltante, más se premia su tendencia a sobreestimar. Cuando el faltante solo pesa el doble (A), sus excedentes pesan más y gana Naïve estacional.

### Reto 1 · Apartado C — Evaluar en test y comparar frente a Naïve
Evalúe en test el modelo seleccionado en cada escenario (la selección ya quedó definida con validación) y Naïve con los mismos costos. Construya una tabla con excedentes, faltantes, costo y diferencia frente a Naïve, y un gráfico de barras. Interprete qué escenario y qué tipo de error explican las diferencias.

In [ ]:
# Se evalúa en test el modelo ya elegido en validación para cada escenario, junto con Naïve, con los mismos costos
filas = []
for esc, (short, over) in ESCENARIOS.items():
    sel = r1_eleccion[esc]
    ev_sel = evaluar(test, cantidades_test[sel], over=over, short=short)
    ev_naive = evaluar(test, cantidades_test["Naive"], over=over, short=short)
    filas.append({"escenario": esc, "seleccionado": sel,
                  "excedentes": ev_sel["unidades_excedentes"], "faltantes": ev_sel["unidades_faltantes"],
                  "costo_test": ev_sel["costo"], "costo_test_Naive": ev_naive["costo"],
                  "diferencia_vs_Naive": ev_naive["costo"] - ev_sel["costo"]})   # > 0 ahorro; < 0 sobrecosto
tabla_r1c = pd.DataFrame(filas).set_index("escenario")
display(tabla_r1c.round(2))

etiquetas = [f"{esc} · {tabla_r1c.loc[esc, 'seleccionado']}" for esc in tabla_r1c.index]
fig = go.Figure([go.Bar(x=etiquetas, y=tabla_r1c["costo_test"], name="Modelo seleccionado"),
                 go.Bar(x=etiquetas, y=tabla_r1c["costo_test_Naive"], name="Naïve")])
fig.update_layout(barmode="group", title="Reto 1 · Costo en test por escenario (£)",
                  xaxis_title="Escenario · modelo seleccionado", yaxis_title="Costo en test (£)")
fig.show()

**Interpretación 1C:**
En el test, **Naïve fue más barato en los tres escenarios**:
- **A:** Naïve estacional costó £5 869,50 frente a £4 949,50 de Naïve (sobrecosto de £920,00).
- **B:** Exponential Smoothing costó £8 957,60 frente a £7 087,90 (sobrecosto de £1 869,70).
- **C:** Exponential Smoothing costó £17 734,25 frente a £13 647,25 (sobrecosto de £4 087,00).

La diferencia la explican los **faltantes**: Naïve estacional dejó 5 549 y Exponential Smoothing 5 884, frente a 4 469 de Naïve, que mantuvo un nivel fijo de 989 unidades por semana y cubrió mejor las semanas normales. Por eso la brecha crece justo en el escenario C, donde cada faltante cuesta £3,00. Lo que en validación era una ventaja (sobreestimar) no ocurrió en el test, porque ahí los modelos subestimaron, sobre todo en la semana del pedido de 4 800 unidades.

### Reto 1 · Apartado D — Recomendar a abastecimiento
Redacte una recomendación para abastecimiento: ¿cambia la selección entre escenarios?, ¿qué método usaría como referencia y por qué?, ¿qué límites tiene la conclusión? Distinga el ahorro simulado de un beneficio real demostrado.

**Decisión 1D:**
Recomendaría a abastecimiento usar **Naïve como referencia** durante la campaña. La selección en validación cambia según los costos (Naïve estacional en A; Exponential Smoothing en B y C), lo que muestra que la recomendación del modelo es sensible al supuesto de costos. Además, en el test ninguno de los modelos seleccionados superó a Naïve en ningún escenario, y la desventaja fue mayor justo en el escenario de campaña (£4 087,00 más en C). Los modelos pueden seguir en evaluación, pero no deberían decidir compras todavía.

**Límites:** los costos de cada escenario son supuestos académicos, no costos reales de la empresa; el test es uno solo, de 8 semanas, y está dominado por un pedido de 4 800 unidades; y los modelos son univariantes, así que no saben que habrá campaña ni cuánto puede subir la demanda. Las diferencias en libras son **costos simulados** bajo esos supuestos, no un ahorro ni una pérdida demostrados.

## Reto 2 — ¿Conviene añadir un colchón?

**Caso.** El área comercial propone planificar unidades adicionales cada semana para reducir faltantes. Almacén advierte que el excedente también cuesta: espacio, capital inmovilizado y posibles liquidaciones. Se debe decidir con evidencia si conviene añadir un colchón al pronóstico.

**Problema.** ¿Qué política de cantidad, sin colchón o con colchón de 0,5; 1 o 1,5 veces el MAE previo, debería usarse, y qué costo produjo en test frente a no usar colchón?

**Datos, supuestos y recursos**
- Modelo seleccionado en la parte guiada (`SELECCIONADO`) y sus pronósticos de validación y test.
- Colchón = ⌈multiplicador × MAE previo⌉; el MAE previo usa solo los 2 bloques de 8 semanas anteriores a cada origen (`mae_previo`, `colchon`, `comparar_politicas`).
- Costos base: £1,50 por faltante y £0,40 por exceso (supuestos académicos). Regla común de selección.

**Apartados**
- **A.** Definir pronóstico, cantidad planificada y políticas
- **B.** Comparar las políticas en validación
- **C.** Seleccionar el multiplicador y evaluarlo en test
- **D.** Recomendar sobre el colchón

**Evidencias que deben quedar en el notebook**
- Tabla del MAE previo y del colchón de cada política por fold.
- Costos por fold y resumen por política con excedentes, faltantes y folds en que el colchón mejoró, más el gráfico de excedentes y faltantes.
- Tabla de test de la política seleccionada frente a la opción sin colchón.
- Interpretaciones A–D.

### Reto 2 · Apartado A — Definir pronóstico, cantidad planificada y políticas
Explique la diferencia entre pronóstico y cantidad planificada. Calcule, para cada fold, el MAE previo (solo con semanas anteriores a su origen) y el colchón que tendría cada política (0; 0,5; 1 y 1,5 × MAE previo).

In [ ]:
# MULTIPLICADORES = [0, 0.5, 1.0, 1.5] ya está definido en la parte guiada
# Para cada fold, el MAE previo usa solo los 2 bloques de 8 semanas anteriores a su origen
filas = []
for k, o in enumerate(origenes, start=1):
    mae_k = mae_previo(SELECCIONADO, train.iloc[:o], f"reto2 · fold {k}")
    fila = {"fold": k, "ultima_semana_calibracion": train.index[o - 1].date(), "inicio_validacion": train.index[o].date(),
            "mae_previo": mae_k}
    fila.update({f"colchón {m:g}×": colchon(m, mae_k) for m in MULTIPLICADORES})
    filas.append(fila)
r2_prep = pd.DataFrame(filas).set_index("fold")
display(r2_prep.round(1))

**Interpretación 2A:**
El **pronóstico** es la estimación de cuánto se va a vender; la **cantidad planificada** es la decisión: pronóstico + colchón. Se pueden equivocar distinto: un buen pronóstico con una mala política sigue costando caro, y viceversa.

El MAE previo de cada fold se calcula con los 2 bloques de 8 semanas anteriores a su origen. En Colab salió 1 400,7 (fold 1), 1 533,2 (fold 2), 1 274,1 (fold 3), 899,9 (fold 4) y 821,3 (fold 5), y la tabla confirma que la última semana de calibración siempre es anterior al inicio de la validación. El MAE previo baja con el tiempo porque los primeros folds se calibran con bloques que incluyen los picos de noviembre y diciembre de 2010. Eso hace que el colchón sea grande: con 0,5× va de unas 411 a 767 unidades por semana, con 1× de unas 822 a 1 534 y con 1,5× de unas 1 232 a 2 300, cuando la mediana semanal del entrenamiento es 584,5.

### Reto 2 · Apartado B — Comparar las políticas en validación
Evalúe las cuatro políticas en los folds de validación con calibración temporal (`comparar_politicas`). Presente el costo por fold, el resumen por política (costo promedio, desviación, excedentes y faltantes), cuántos folds mejoró cada colchón frente a no usarlo y un gráfico de excedentes y faltantes.

In [ ]:
r2_folds = comparar_politicas(etiqueta="reto2 · validación")
r2_resumen = resumir(r2_folds, "multiplicador", MULTIPLICADORES)

# Costo por fold de cada política y número de folds en que cada colchón costó menos que no usarlo (0×)
costo_fold = r2_folds.pivot(index="fold", columns="multiplicador", values="costo")
costo_fold.columns = [f"{m:g}×" for m in costo_fold.columns]
display(costo_fold.round(1))
r2_resumen["folds_mejor_que_0x"] = [np.nan if m == 0 else int((costo_fold[f"{m:g}×"] < costo_fold["0×"]).sum())
                                    for m in MULTIPLICADORES]
display(r2_resumen[["costo_prom", "costo_sd", "MAE_prom", "excedentes", "faltantes", "folds_mejor_que_0x"]].round(2))

fig = go.Figure([go.Bar(x=[f"{m:g}×" for m in MULTIPLICADORES], y=r2_resumen["excedentes"], name="Unidades excedentes"),
                 go.Bar(x=[f"{m:g}×" for m in MULTIPLICADORES], y=r2_resumen["faltantes"], name="Unidades faltantes")])
fig.update_layout(barmode="group", title="Reto 2 · Excedentes y faltantes en validación por política (5 folds)",
                  xaxis_title="Multiplicador del MAE previo", yaxis_title="Unidades")
fig.show()

**Interpretación 2B:**
Cuanto más colchón, más caro sale en promedio:
- **0×:** £4 324,10 (desviación £2 335,11).
- **0,5×:** £4 806,72 (£1 784,08).
- **1×:** £5 833,74 (£1 661,61).
- **1,5×:** £7 674,08 (£2 000,88).

El colchón cumple lo que promete con los faltantes (6 167 → 2 440 → 149 → 0), pero los excedentes crecen mucho más rápido (30 925 → 50 934 → 72 363 → 95 926). Solo en un fold (el 2, donde Exponential Smoothing había subestimado) el colchón de 0,5× y el de 1× costaron menos que no usarlo; el de 1,5× no mejoró en ninguno. Con colchón el costo varía un poco menos entre folds, pero esa menor variabilidad se paga con un costo promedio más alto. Como el modelo ya sobreestima en la mayoría de los periodos, el colchón suele proteger contra faltantes que no iban a ocurrir.

### Reto 2 · Apartado C — Seleccionar el multiplicador y evaluarlo en test
Seleccione el multiplicador con la regla común (sin mirar el test). Calcule el colchón del test con el MAE previo del entrenamiento y evalúe la política seleccionada frente a la opción sin colchón: costo, excedentes y faltantes. Compruebe si coincide con la política fijada en la sección 10.

In [ ]:
r2_mult = seleccionar(r2_resumen)
# Colchón del test: MAE previo calculado solo con el entrenamiento (misma regla de 2 bloques)
mae_t = mae_previo(SELECCIONADO, train, "reto2 · test")
colchon_t = colchon(r2_mult, mae_t)
tabla_r2c = pd.DataFrame({
    f"{SELECCIONADO} sin colchón": evaluar(test, cantidades_test[SELECCIONADO]),
    f"Política seleccionada: {SELECCIONADO} + {r2_mult:g} × MAE previo (colchón {colchon_t})":
        evaluar(test, cantidades_test[SELECCIONADO] + colchon_t)}).T
display(tabla_r2c.round(2))
print(f"Multiplicador seleccionado en validación: {r2_mult:g} · MAE previo del test = {mae_t:.1f} · colchón = {colchon_t} unidades por semana")
print("¿Coincide con la política fijada en la sección 10?", "Sí" if r2_mult == MULT else "No")

**Interpretación 2C:**
La regla común selecciona el multiplicador **0** (sin colchón): es el de menor costo promedio (£4 324,10) y el único candidato, porque el límite del 1 % es £4 367,34. El colchón del test es 0 unidades (MAE previo del entrenamiento = 618,8), así que la política seleccionada y la opción sin colchón son idénticas en el test: £8 957,60, con 329 excedentes y 5 884 faltantes. **Coincide** con la política fijada en la sección 10.

### Reto 2 · Apartado D — Recomendar sobre el colchón
Recomiende si corresponde añadir colchón y fundamente con la validación. Explique por qué la cantidad objetivo semanal no equivale automáticamente a una cantidad a comprar (inventario utilizable, pedidos que llegarán a tiempo y arrastre entre semanas).

**Decisión 2D:**
**No corresponde añadir colchón** con este modelo. En validación, no usar colchón fue lo más barato en promedio y ganó en 4 de los 5 folds; cada colchón probado aumentó el costo promedio entre £482,62 (0,5×) y £3 349,98 (1,5×). Retrospectivamente, en el test un colchón de 0,5× habría bajado el costo a £7 247,80, pero solo por el pedido extraordinario de 4 800 unidades; aun así habría seguido por encima de Naïve (£7 087,90). Es un resultado mirado después y no justifica cambiar la política.

La **cantidad objetivo semanal** no es una cantidad a comprar. Para pasar a una compra hay que restar el **inventario utilizable** (apto para la venta) y los **pedidos que llegarán a tiempo**, y considerar el **arrastre entre semanas**: lo que sobra una semana reduce lo que se necesita la siguiente. Por eso sumar los objetivos semanales sobreestima lo que realmente habría que pedir.

### Reto 2 · Ampliación opcional: ventana de calibración
Repita la comparación con 1, 2 y 3 bloques de calibración y observe si cambia el multiplicador elegido. Si muestra costos de test de las políticas descartadas, identifíquelos como retrospectivos: no se usan para cambiar la política.

In [ ]:
# Opcional: para b en [1, 2, 3], use comparar_politicas(bloques=b, etiqueta=...) , resumir(...) y seleccionar(...)
filas = []
for b in [1, 2, 3]:
    res_b = resumir(comparar_politicas(bloques=b, etiqueta=f"reto2 · ventana {b}"), "multiplicador", MULTIPLICADORES)
    m_b = seleccionar(res_b)
    filas.append({"bloques_calibracion": b, "multiplicador_elegido": m_b,
                  **{f"costo_prom {m:g}×": res_b.loc[m, "costo_prom"] for m in MULTIPLICADORES}})
ventanas = pd.DataFrame(filas).set_index("bloques_calibracion")
display(ventanas.round(2))

# RETROSPECTIVO: costo en test de todas las políticas (2 bloques). Solo informativo; no cambia la política fijada.
retro = pd.DataFrame({f"{m:g}×": evaluar(test, cantidades_test[SELECCIONADO] + colchon(m, mae_t)) for m in MULTIPLICADORES}).T
retro["colchón"] = [colchon(m, mae_t) for m in MULTIPLICADORES]
print("Costos de test RETROSPECTIVOS (no se usan para cambiar la política):")
display(retro[["colchón", "unidades_excedentes", "unidades_faltantes", "costo"]].round(2))

**Comentario opcional 2:**
Con 1, 2 y 3 bloques de calibración se elige siempre el multiplicador 0: cambiar la ventana mueve un poco el costo de los colchones, pero en ningún caso alguno queda por debajo de no usar colchón. La decisión no depende de la ventana elegida. Los costos de test de las políticas descartadas (0,5× ≈ £7 247,80; 1× ≈ £7 649,50; 1,5× ≈ £8 052,50) son **retrospectivos**: se muestran solo como referencia y no cambian la política fijada.

## Reto 3 — ¿Funciona la misma solución para otro producto?

**Caso.** La empresa quiere extender el procedimiento a un segundo producto de alta rotación antes de aplicarlo a toda la cartera. La gerencia pregunta si el método que se eligió para el primer producto puede generalizarse.

**Problema.** ¿Qué modelo se selecciona para el segundo producto con las mismas reglas, cómo le fue en test frente a Naïve y qué haría la empresa con cada producto?

**Datos, supuestos y recursos**
- Segundo SKU de la lista de elegibles, definida solo con el entrenamiento (no se elige mirando el test).
- Mismas semanas completas, limpieza, backtesting, costos base y regla común de selección.
- Se comparan modelos **sin colchón** para mantener el alcance claro; la política de colchón se trabajó en el Reto 2.

**Apartados**
- **A.** Identificar el producto y describir su serie
- **B.** Backtesting y selección con la regla común
- **C.** Evaluar en test y comparar ambos productos
- **D.** Recomendación consolidada

**Evidencias que deben quedar en el notebook**
- Serie del segundo producto y tabla descriptiva del entrenamiento frente al primero, con su gráfico.
- Tabla de backtesting por modelo y seleccionado.
- Tabla comparativa de ambos productos con modelo, costos, ahorro o sobrecosto y resultado.
- Interpretaciones A–D.

### Reto 3 · Apartado A — Identificar el producto y describir su serie
Identifique el segundo SKU elegible, construya su serie con las mismas semanas completas y separe entrenamiento y test. Describa su comportamiento solo con el entrenamiento y compárelo con el primer producto (tabla y gráfico).

In [ ]:
assert len(elegibles) >= 2, "Se necesita un segundo SKU elegible"
SKU2 = elegibles.loc[1, "StockCode"]
# Serie del segundo producto con la misma construcción de la sección 6 (mismas semanas completas y ceros explícitos)
desc2 = uk.loc[uk["StockCode"] == SKU2, "Description"].dropna().astype(str).mode().iloc[0]
serie2 = (uk[uk["StockCode"] == SKU2].groupby("week")["Quantity"].sum()
          .reindex(semanas, fill_value=0).astype(float))
serie2.name = "unidades"
train2, test2 = serie2.loc[semanas_train], serie2.loc[semanas_test]
print("Segundo SKU:", SKU2, "·", desc2)

def describir(y):
    """Estadísticos descriptivos de una serie de entrenamiento."""
    return {"unidades": y.sum(), "media": y.mean(), "mediana": y.median(), "desv_est": y.std(),
            "coef_variacion": y.std() / y.mean(), "maximo": y.max(), "semanas_sin_venta": int((y == 0).sum()),
            "media_2010": y[y.index.year == 2010].mean(), "media_2011": y[y.index.year == 2011].mean(),
            "media_ultimas_8": y.iloc[-8:].mean()}

r3_desc = pd.DataFrame({f"{SKU} · {desc}": describir(train), f"{SKU2} · {desc2}": describir(train2)}).T
display(r3_desc.round(2))

fig = go.Figure([go.Scatter(x=train.index, y=train.values, name=f"{SKU} · {desc}"),
                 go.Scatter(x=train2.index, y=train2.values, name=f"{SKU2} · {desc2}")])
fig.update_layout(title="Reto 3 · Ventas semanales registradas en el entrenamiento",
                  xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

**Interpretación 3A:**
El segundo SKU elegible es **85123A · WHITE HANGING HEART T-LIGHT HOLDER**, elegido con la lista definida solo con el entrenamiento. Comparado con el primero (84077):
- Vende parecido en volumen (80 903 frente a 88 273 unidades), pero es mucho más **estable**: desviación de 582,80 frente a 1 080,36 y coeficiente de variación de 0,69 frente a 1,17.
- Su pico máximo es menor (3 329 frente a 7 063) y su mediana (718,5) está más cerca de la media (842,74), así que depende menos de pedidos extraordinarios.
- Tiene 2 semanas sin ventas (las de posible cierre) frente a 3.
- Su nivel cae más en 2011: la media pasa de 977,12 en 2010 a 641,52 en 2011 (−34 %), y en las últimas 8 semanas del entrenamiento promedia 460 unidades (653,25 en el primer producto).

Es una serie más regular, pero con un nivel decreciente.

### Reto 3 · Apartado B — Backtesting y selección con la regla común
Ejecute el mismo backtesting con Naïve, Naïve estacional y Exponential Smoothing (`backtest`), resuma los resultados y seleccione con la regla común. Presente la tabla e indique si la selección se decidió por la tolerancia del 1 %.

In [ ]:
filas2, _ = backtest(train2, MODELOS, "reto3")
tabla2 = resumir(filas2, "modelo", MODELOS)
SELECCIONADO2 = seleccionar(tabla2)
# Marca qué modelos quedaron dentro de la tolerancia del 1 % (candidatos de la regla común)
tabla2["candidato_1pct"] = tabla2["costo_prom"] <= tabla2["costo_prom"].min() * (1 + TOLERANCIA) + 1e-9
display(tabla2.round(2))
print("Candidatos dentro del 1 %:", list(tabla2.index[tabla2["candidato_1pct"]]))
print("MODELO SELECCIONADO PARA EL SEGUNDO PRODUCTO (antes del test):", SELECCIONADO2)

**Interpretación 3B:**
Con la misma regla común se selecciona **Exponential Smoothing**: costo promedio de validación de £2 591,66 (desviación £1 906,87), frente a £3 306,84 de Naïve estacional y £4 814,82 de Naïve. Fue el único candidato (límite del 1 % ≈ £2 617,58), así que la selección no se decidió por la tolerancia.

Igual que en el primer producto, el ranking por MAE no coincide con el de costo: Naïve estacional tiene un MAE ligeramente menor (459,32 frente a 474,88), pero más faltantes (8 350 frente a 4 873), y eso lo hace más caro.

### Reto 3 · Apartado C — Evaluar en test y comparar ambos productos
Evalúe en test el modelo seleccionado y Naïve para el segundo producto. Construya una tabla comparativa de ambos productos (sin colchón) con modelo seleccionado, costo de validación, costo de test del seleccionado y de Naïve, ahorro o sobrecosto, folds en que superó a Naïve y resultado de la evaluación, aplicando la regla de adopción de la sección 13.

In [ ]:
# Segundo producto: pronóstico de test con el modelo seleccionado y con Naïve, entrenando con train2
q2_sel = pronosticar(SELECCIONADO2, train2, TEST_H, "reto3 · test")
q2_naive = pronosticar("Naive", train2, TEST_H, "reto3 · test")
ev2_sel, ev2_naive = evaluar(test2, q2_sel), evaluar(test2, q2_naive)
bt2 = pd.DataFrame(filas2)

def resultado_adopcion(costo_sel, costo_nv, folds, modelo):
    """Regla de adopción de la sección 13."""
    if modelo == "Naive":
        return "El seleccionado es el propio Naïve: se mantiene Naïve como referencia"
    if costo_sel < costo_nv and folds > len(origenes) / 2:
        return "Adoptar como piloto controlado frente a Naïve"
    if costo_sel < costo_nv:
        return "Mantener en evaluación"
    return "No adoptar como sustituto de Naïve; Naïve se mantiene como referencia"

def folds_vs_naive(bt_df, modelo):
    """Folds de validación en que el modelo costó menos que Naïve."""
    c = bt_df.pivot(index="fold", columns="modelo", values="costo")
    return int((c[modelo] < c["Naive"]).sum())

# Primer producto (sin colchón): objetos de la parte guiada
c1_sel = res_test.loc[f"{SELECCIONADO} (sin colchón)", "costo"]
c1_nv = res_test.loc["Naive (sin colchón)", "costo"]
f1 = folds_vs_naive(bt, SELECCIONADO)
# Segundo producto
f2 = folds_vs_naive(bt2, SELECCIONADO2)

comparacion = pd.DataFrame([
    {"producto": f"{SKU} · {desc}", "modelo_seleccionado": SELECCIONADO,
     "costo_validacion_prom": resumen_val.loc[SELECCIONADO, "costo_prom"],
     "costo_test_seleccionado": c1_sel, "costo_test_Naive": c1_nv, "ahorro_vs_Naive": c1_nv - c1_sel,
     "folds_supero_Naive": f"{f1} de {len(origenes)}", "resultado": resultado_adopcion(c1_sel, c1_nv, f1, SELECCIONADO)},
    {"producto": f"{SKU2} · {desc2}", "modelo_seleccionado": SELECCIONADO2,
     "costo_validacion_prom": tabla2.loc[SELECCIONADO2, "costo_prom"],
     "costo_test_seleccionado": ev2_sel["costo"], "costo_test_Naive": ev2_naive["costo"],
     "ahorro_vs_Naive": ev2_naive["costo"] - ev2_sel["costo"],
     "folds_supero_Naive": f"{f2} de {len(origenes)}", "resultado": resultado_adopcion(ev2_sel["costo"], ev2_naive["costo"], f2, SELECCIONADO2)},
]).set_index("producto")
display(comparacion.round(2))
print(f"Segundo producto en test · {SELECCIONADO2}: excedentes {ev2_sel['unidades_excedentes']:.0f}, faltantes {ev2_sel['unidades_faltantes']:.0f} · "
      f"Naïve: excedentes {ev2_naive['unidades_excedentes']:.0f}, faltantes {ev2_naive['unidades_faltantes']:.0f}")

**Interpretación 3C:**
Los resultados son opuestos:
- **Segundo producto (85123A):** Exponential Smoothing costó £3 376,40 en el test frente a £6 640,70 de Naïve, un **ahorro simulado de £3 264,30 (49,2 %)**, y superó a Naïve en los 5 folds de validación. Con la regla de la sección 13, el resultado es **adoptar como piloto controlado**. Naïve repitió una semana baja y dejó 4 425 unidades faltantes; Exponential Smoothing dejó 2 036 faltantes y 806 excedentes.
- **Primer producto (84077):** el mismo método costó £8 957,60 frente a £7 087,90 de Naïve (sobrecosto de £1 869,70), aunque había ganado 4 de 5 folds; el resultado es **no adoptar**.

El mismo procedimiento funcionó en la serie más estable y falló en la que depende de pedidos extraordinarios.

### Reto 3 · Apartado D — Recomendación consolidada
Prepare una recomendación para la empresa: qué haría con cada producto (adoptar, mantener en evaluación o no adoptar), qué evidencia la respalda y por qué dos productos no permiten generalizar el método a toda la cartera.

**Decisión 3D:**
- **Producto 84077 (WORLD WAR 2 GLIDERS):** **no adoptar** Exponential Smoothing; Naïve sigue como referencia. Ganó en validación, pero costó 26,4 % más en el test, en una serie muy irregular y con pedidos grandes difíciles de anticipar.
- **Producto 85123A (WHITE HANGING HEART T-LIGHT HOLDER):** **piloto controlado** con Exponential Smoothing. Ganó a Naïve en los 5 folds y en el test (ahorro simulado de £3 264,30). El piloto debe comprobar en semanas nuevas, con la misma regla, que la reducción de costo se mantiene antes de decidir compras con él.

**Por qué no se puede generalizar:** con dos productos el método dio dos decisiones distintas. Son solo 2 de los 389 SKUs elegibles, ambos de los más vendidos (no representan a los de baja rotación), y cada uno tiene un solo test de 8 semanas. Para extender el procedimiento a la cartera habría que evaluar producto por producto, o por grupos con comportamiento parecido, y en más ventanas de prueba.

### Revisión de advertencias y verificaciones automáticas

Se resumen las advertencias de **todos** los ajustes de Exponential Smoothing (backtesting, calibración, test, reentrenamiento final y retos) y se comprueban las condiciones de validez del flujo.

In [ ]:
aj = pd.DataFrame(registro_ajustes)
display(aj.groupby("etapa").agg(ajustes=("ajuste", "count"), con_advertencias=("advertencias", lambda s: int((s > 0).sum())),
                                tipos=("tipos", lambda s: "; ".join(sorted({t for x in s for t in x.split("; ") if t}))))
          .sort_values("ajustes", ascending=False))
mensajes = sorted({m for x in aj["mensajes"] for m in x.split(" | ") if m})
print("Mensajes distintos:", mensajes if mensajes else "ninguno")
print("Especificaciones usadas:", aj["especificacion"].value_counts().to_dict())

verificaciones = {
    "Ningún fold invade el test": all(train.index[o + BT_H - 1] < semanas_test[0] for o in origenes),
    "Colchón de cada fold calibrado solo con semanas anteriores a su origen": bool((pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all()),
    "Colchón del test calculado solo con el entrenamiento": len(train) == len(semanas_train) and train.index[-1] < semanas_test[0],
    "La tabla final usa el multiplicador fijado": COLCHON_FINAL == colchon(MULT, MAE_FINAL),
    "El costo de la política en test se recalcula con su colchón": np.isclose(costo_pol, evaluar(test, q_politica)["costo"]),
}
for k, v in verificaciones.items():
    print(("OK   " if v else "FALLA"), k)

,ajustes,con_advertencias,tipos
etapa,,,
calibración,10,0,
backtesting,5,0,
calibración final,2,0,
calibración test,2,0,
final,1,0,
test,1,0,


Mensajes distintos: ninguno
Especificaciones usadas: {'nivel + tendencia + estacionalidad aditivas (m = 4)': 21}
OK    Ningún fold invade el test
OK    Colchón de cada fold calibrado solo con semanas anteriores a su origen
OK    Colchón del test calculado solo con el entrenamiento
OK    La tabla final usa el multiplicador fijado
OK    El costo de la política en test se recalcula con su colchón


### Retroalimentación final

**Pregunta 10.** Indique dos razones por las que el modelo podría fallar en producción aun cuando el backtesting haya sido bueno.

**Respuesta:**
1. **Cambios en la demanda que el modelo no ve:** las ventas dependen mucho de pedidos grandes de pocos clientes mayoristas (en el test, un solo pedido fue el 90 % de la semana pico). Un modelo univariante solo mira las ventas pasadas, así que una campaña, un cliente nuevo o uno que deja de comprar cambian la serie sin aviso.
2. **El pasado evaluado no representa al futuro:** el backtesting cubrió de enero a octubre de 2011 y el entrenamiento solo tiene un otoño. Si la temporada navideña, los cierres de fin de año o el nivel de ventas cambian, el modelo proyecta un patrón que ya no aplica. A eso se suma que los ceros pueden ser faltas de inventario y no demanda nula, y que los costos reales pueden ser distintos a los supuestos con los que se eligió el modelo.

# 15. Ticket de salida

**«El modelo seleccionado antes del test fue __________, con la política __________ (costo promedio de validación £______). En el test costó £______ frente a £______ de Naïve. Mi decisión es __________ (adoptar / mantener en evaluación / no adoptar) porque ____________________. La proyección académica de ______ unidades no es una orden de compra porque ____________________.»**

**Respuesta:**
«El modelo seleccionado antes del test fue **Exponential Smoothing (nivel + tendencia + estacionalidad aditiva, m = 4)**, con la política **ExpSmoothing + 0 × MAE previo, es decir, sin colchón** (costo promedio de validación **£4 324,10**). En el test costó **£8 957,60** frente a **£7 087,90** de Naïve. Mi decisión es **no adoptar** porque **en el test costó £1 869,70 más que Naïve (26,4 %) por subestimar la demanda, aunque había ganado 4 de 5 folds de validación**. La proyección académica de **9 291** unidades no es una orden de compra porque **es la suma de objetivos semanales que no descuenta el inventario utilizable, los pedidos que llegarán a tiempo ni el arrastre de excedentes entre semanas**.»

# 16. Conclusiones

Registre tres conclusiones de carácter técnico y de negocio.

1. **Ganar en validación no asegura ganar en producción.** Exponential Smoothing fue el mejor en validación (£4 324,10 y 4 de 5 folds a su favor), pero en el test costó 26,4 % más que Naïve. Por eso la decisión se separa del modelo con menor error y siempre se compara con un baseline simple.
2. **El costo asimétrico cambia qué modelo conviene.** Los rankings por MAE/RMSE y por costo salieron invertidos, y en el Reto 1 la selección cambió según el peso del faltante. Con faltantes caros, lo que importa es la dirección del error, no solo su tamaño. Además, el colchón no ayudó porque el modelo ya sobreestimaba.
3. **El mismo método no sirve igual para todos los productos.** En el producto estable (85123A) Exponential Smoothing ahorró 49,2 % frente a Naïve; en el irregular (84077), dominado por un pedido de 4 800 unidades, perdió. Cada SKU necesita su propia evaluación, y las proyecciones deben pasar por inventario y pedidos en tránsito antes de convertirse en compras.

# 17. Evidencia de entrega

- [x] Fuente, hojas, dimensiones, rango de fechas y versiones visibles.
- [x] Limpieza documentada (incluidos `StockCode` nulos).
- [x] Semanas de lunes a domingo; semanas incompletas excluidas.
- [x] Test de 8 semanas reservado antes de seleccionar el SKU, el modelo y la política.
- [x] SKU seleccionado con criterios previos al test y límites del criterio de disponibilidad explicados.
- [x] Exploración solo con entrenamiento.
- [x] Backtesting de 8 semanas con fechas de cada fold y comprobación de que no invade el test.
- [x] Regla única de selección aplicada a modelo, política y retos.
- [x] Colchón calibrado solo con errores previos a cada origen.
- [x] Política fijada antes del test y evaluada una vez.
- [x] Decisión de adopción separada de la proyección académica.
- [x] Tres retos individuales con sus apartados A, B, C y D resueltos e interpretados; advertencias revisadas.
- [x] Notebook en GitHub (`semana10/`) y en su propio Colab; ambos URLs presentados en la actividad de la semana.
- [x] Uso de IA declarado.

# Referencias
- Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts.
- Sharda, R., Delen, D., & Turban, E. *Business Intelligence, Analytics, Data Science and AI*. Pearson.